In [22]:
import os
import gradio as gr
from openai import OpenAI
from kokoro import KPipeline
import soundfile as sf
import numpy as np
import whisper
import torch

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key :
    print(f"key found");

MODEL = "llama3:latest"
openai = OpenAI(base_url=openai_api_key,api_key="ollama")    


key found


In [ ]:
# Initialize Voice Pipeline (English - American)
# Kokoro is extremely efficient on your 3060
pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')

In [25]:
# Load Whisper (Tiny is incredibly fast on a 3060)
stt_model = whisper.load_model("base", device="cuda")

def chat_and_speak(message, history):
    """
    Handles the LLM logic and Voice generation.
    'history' here is a list of dicts: [{'role': 'user', 'content': '...'}, ...]
    """
    # Get Response from Ollama
    response = openai.chat.completions.create(
        model=MODEL, 
        messages=history + [{'role': 'user', 'content': message}]
    )
    text_response = response.choices[0].message.content
    
    # Generate Voice via Kokoro
    generator = pipeline(text_response, voice='am_michael', speed=1)
    audio_segments = [audio for _, _, audio in generator]
    full_audio = np.concatenate(audio_segments) if audio_segments else None

    # Return only the bot's text and the audio for now
    return text_response, (24000, full_audio)


def process_all_inputs(text_input, audio_input, chat_history, voice_name):
    # 1. Determine the user's message
    user_message = ""
    
    if text_input:
        user_message = text_input
    elif audio_input:
        # Transcribe using Whisper (as we set up before)
        result = stt_model.transcribe(audio_input)
        user_message = result["text"]
    else:
        # Neither input was provided
        return "", chat_history, None

    # 2. Add user message to history (Messages format for Gradio 5)
    chat_history.append({"role": "user", "content": user_message})

    # 3. Get AI response (using your existing chat_and_speak logic)
    bot_text, audio_data = chat_and_speak(user_message, chat_history)
    
    # 4. Add bot response to history
    chat_history.append({"role": "assistant", "content": bot_text})
    
    # Return: (clear textbox, updated history, audio response)
    return "", chat_history, audio_data    


def transcribe_and_chat(audio_path, chat_history, voice_name):
    if audio_path is None:
        return chat_history, None

    # 1. Transcribe the microphone audio
    result = stt_model.transcribe(audio_path)
    user_text = result["text"]

    # 2. Add user message to history
    chat_history.append({"role": "user", "content": user_text})

    # 3. Get AI Text and Voice (using your existing function)
    bot_text, audio_data = chat_and_speak(user_text, chat_history)
    
    # 4. Add bot response to history
    chat_history.append({"role": "assistant", "content": bot_text})
    
    return chat_history, audio_data    

In [ ]:
# Gradio UI with Blocks
with gr.Blocks() as demo:
    gr.Markdown("Voice Chatbot")
    
    # Use 'messages' type for 2026 Gradio standards
    chatbot = gr.Chatbot(type="messages", label="Conversation")
    msg = gr.Textbox(label="Ask me anything...", placeholder="Type here and press Enter")
    audio_output = gr.Audio(label="AI Voice", autoplay=True)
    
    def respond(user_input, chat_history):
        # 1. Add user message to history immediately
        chat_history.append({"role": "user", "content": user_input})
        
        # 2. Call the AI logic
        bot_text, audio = chat_and_speak(user_input, chat_history)
        
        # 3. Add bot message to history
        chat_history.append({"role": "assistant", "content": bot_text})
        #print(chat_history)
        # Return: (empty textbox, updated history, audio)
        return "", chat_history, audio

    # Trigger on Enter/Submit
    msg.submit(respond, [msg, chatbot], [msg, chatbot, audio_output])

In [ ]:
demo.launch()

How to use my mic instead of writing?

Instead of just a msg.submit from a Textbox, we will use Gradio’s gr.Audio(sources=["microphone"]).
The Model: openai/whisper-tiny or base. 
On a 3060, these transcribe almost instantly.
The Flow: 
1. Record audio in Gradio.
2. Whisper converts Audio $\rightarrow$ Text.
3. Text goes to Llama 3.2 (Ollama).
4. Llama response goes to Kokoro (Voice).+1

uv add openai-whisper
If you don't have ffmpeg on Pop!_OS:
sudo apt install ffmpeg

In [26]:
# Gradio UI Update
with gr.Blocks() as demo:
    gr.Markdown("# RTX 3060 Voice & Text Chatbot")
    chatbot = gr.Chatbot(type="messages")
    with gr.Row():
        msg = gr.Textbox(label="Type here...", placeholder="Press Enter to send")
        mic = gr.Audio(sources=["microphone"], type="filepath", label="Or speak here")
        voice_select = gr.Dropdown(choices=['am_michael', 'af_bella'], value='am_michael', label="Voice")
    
    audio_output = gr.Audio(label="AI Response", autoplay=True)

    # EVENT 1: User presses Enter in Textbox
    msg.submit(process_all_inputs, [msg, mic, chatbot, voice_select], [msg, chatbot, audio_output])

    # EVENT 2: User stops recording on Microphone

    mic.stop_recording(process_all_inputs, [msg, mic, chatbot, voice_select], [msg, chatbot, audio_output])
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


/home/billysvk/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/gradio/processing_utils.py:688: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
